In [ ]:
# Install Node.js directly into the notebook container environment to solve the JS Runtime error
!apt-get update && apt-get install -y nodejs

# Force a clean install of your essential video-processing framework blocks
!pip install -q yt-dlp "yt-dlp[default]" faster-whisper transformers accelerate bitsandbytes moviepy
!pip install -U --no-deps decorator


In [ ]:
import os
import re
import gc
import torch
import yt_dlp
from faster_whisper import WhisperModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# --- 1. DOWNLOAD YOUTUBE VIDEO ---
def download_video(url, output_name="input_video.mp4"):
    print("🎬 Downloading video from YouTube...")
    ydl_opts = {
        'format': 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best',
        'outtmpl': output_name,
        'quiet': True,
        'js_runtimes': {'node': {}}, 
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])
    print("✅ Video download finished successfully.")
    return output_name

# --- 2. TRANSCRIPTION PIPELINE (GPU 0) ---
def transcribe_video(video_path):
    print("🎙️ Transcribing video on GPU 0...")
    model = WhisperModel("medium", device="cuda", device_index=0, compute_type="float16")
    segments, info = model.transcribe(video_path, beam_size=1)
    
    formatted_transcript = ""
    for segment in segments:
        formatted_transcript += f"[{segment.start:.1f}s - {segment.end:.1f}s]: {segment.text}\n"
    
    del model
    gc.collect()
    torch.cuda.empty_cache()
    print("✅ Transcription complete. GPU 0 cleared.")
    return formatted_transcript

# --- 3. EXTRACT & SORT 5 COMPLETE VIRAL HOOKS BY PROBABILITY (GPU 1) ---
def find_five_viral_hooks(transcript):
    print("🧠 Analyzing transcript for 5 prioritized complete viral hooks using Qwen2.5-3B on GPU 1...")
    model_name = "Qwen/Qwen2.5-3B-Instruct" 
    
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, 
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4"
    )
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name, 
        quantization_config=bnb_config, 
        device_map={"": "cuda:1"},
        attn_implementation="sdpa",
        low_cpu_mem_usage=True
    )
    
    lines = transcript.split('\n')
    if len(lines) > 400:
        print(f"⚠️ Transcript too long. Truncating lines to fit safely inside VRAM...")
        transcript = "\n".join(lines[:400])
        
    prompt = f"""
    You are an expert viral content editor. Review this video transcript and identify 5 separate, distinct, non-overlapping 30 to 60-second clips that have strong standalone hooks.
    Calculate a Virality Score (0-100) for each clip based on emotional impact, clarity, and pacing. 
    
    CRITICAL RULE FOR COMPLETENESS:
    Each clip MUST contain a complete message or standalone thought. The video must start at the exact beginning of a sentence and end immediately after a sentence is fully finished. Do NOT select a start or end timestamp that cuts off a word, splits a phrase, or leaves a sentence hanging in the middle.
    
    Sort the final output list strictly from HIGHEST virality score to LOWEST virality score.
    
    Respond ONLY with the start and end timestamps in this exact markdown format for each clip:
    RANK 1 [SCORE: x] - START: [seconds] | END: [seconds]
    RANK 2 [SCORE: x] - START: [seconds] | END: [seconds]
    RANK 3 [SCORE: x] - START: [seconds] | END: [seconds]
    RANK 4 [SCORE: x] - START: [seconds] | END: [seconds]
    RANK 5 [SCORE: x] - START: [seconds] | END: [seconds]
    Do not write any introductory or explanatory text.

    Transcript:
    {transcript}
    """
    
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt", truncation=True, max_length=3500).to("cuda:1")
    
    with torch.no_grad():
        generated_ids = model.generate(**model_inputs, max_new_tokens=250)
        
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    
    # 💡 CRITICAL FIX: Access index 0 of the decoded list BEFORE running the split method
    raw_output_string = response[0]
    output_text = raw_output_string.split("<|im_start|>assistant\n")[-1]
    
    print("\n📊 LLM Generation Strategy Order:\n", output_text.strip())
    
    del model
    del model_inputs
    gc.collect()
    torch.cuda.empty_cache()
    return output_text

# --- 4. MULTI-CLIP VERTICAL RENDERING LOOP VIA FFMPEG ---
def crop_multiple_shorts(video_path, llm_output):
    print("\n✂️ Parsing pipeline outputs and batch processing clips...")
    
    lines = llm_output.strip().split('\n')
    matches = []
    for line in lines:
        time_match = re.search(r'START:\s*([\d\.]+)\s*\|\s*END:\s*([\d\.]+)', line, re.IGNORECASE)
        score_match = re.search(r'SCORE:\s*([\d\.]+)', line, re.IGNORECASE)
        
        if time_match:
            start = float(time_match.group(1))
            end = float(time_match.group(2))
            score = score_match.group(1) if score_match else "N/A"
            matches.append((start, end, score))
            
    if not matches:
        print("⚠️ Failed to parse structured timestamps. Running fallback sequential mode.")
        matches = [(10.0, 50.0, "75"), (60.0, 100.0, "70"), (120.0, 160.0, "65"), (180.0, 220.0, "60"), (240.0, 280.0, "55")]
        
    filter_chain = "crop=ih*(9/16):ih:(iw-ow)/2:0"
    
    for idx, (start_time, end_time, score) in enumerate(matches[:5]):
        duration = end_time - start_time
        output_filename = f"viral_rank_{idx + 1}_score_{score}.mp4"
        
        print(f"🎬 Rendering Clip {idx + 1}/5 (Virality Score: {score}): Start={start_time}s | Duration={duration}s -> {output_filename}")
        
        ffmpeg_cmd = f'ffmpeg -y -ss {start_time} -i "{video_path}" -t {duration} -vf "{filter_chain}" -c:v libx264 -c:a aac -strict -2 "{output_filename}" > /dev/null 2>&1'
        os.system(ffmpeg_cmd)
        
    print(f"\n🎉 Prioritized complete batch rendering finished! Check your working directory.")

# --- RUN THE ENGINE OVER 5 BATCH GENERATIONS ---
VIDEO_URL = "https://youtu.be/sMMNavdrIEc"

video_file = download_video(VIDEO_URL)
transcript_data = transcribe_video(video_file)
chosen_hooks_plan = find_five_viral_hooks(transcript_data)
crop_multiple_shorts(video_file, chosen_hooks_plan)
